In [2]:
import os
import sys
import pandas as pd
import json
import ast

In [ ]:
pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 150)

In [3]:
current_dir = os.getcwd()

# 프로젝트 루트 디렉토리 찾기 (notebooks/preprocessing에서 두 단계 위로 이동)
root_dir = os.path.abspath(os.path.join(current_dir, "..", ".."))
print(f"프로젝트 루트 디렉토리: {root_dir}")
sys.path.append(root_dir)


프로젝트 루트 디렉토리: /Users/1112436/Desktop/Project/prec/map-search-agent


In [4]:
data_root = os.path.join(root_dir, 'data', 'raw', 'prod_info.json')

In [238]:
with open(data_root, 'r') as file:
    meta_data = json.load(file)

meta_data = pd.json_normalize(meta_data)

In [242]:
result = []
for col_nm in meta_data.columns:
    if col_nm.startswith("commonRule"):
        result.append(col_nm)
    
print(result)

['commonRule.lineSuspensionCondition.value', 'commonRule.planChangeCondition.dailyPlanChangeLimit.value', 'commonRule.planChangeCondition.monthlyPlanChangeLimit.value']


# 합의된 필드 제거

In [250]:
drop_cols = [
    'productId'
    'providingKind.nonDiyVsDiy.value', 
    'managementInfo.approvalInfo.value', 
    'managementInfo.versionInfo.value',
    'managementInfo.productId.value',
    'managementInfo.productRequiredNotice.value',
    'managementInfo.productOperationPeriod.value',
    'commonRule.lineSuspensionCondition.value',
    'commonRule.smsSendingLimitThreshold',
    'commonRule.legalRepresentativeChangeEligibility',
    'commonRule.seniorDataExceedLimitRule',
    'commonRule.generalDataExceedLimitRule',
    'commonRule.fairUserPolicyRule',
    'commonRule.dataExhaustionTimeOrder',
    'campaignRelation|signupPreSignup',
    'campaignRelation.signupPreTermination.campaignInformationDTO.campaignList', 
    'campaignRelation.signupConcurrentTermination.campaignInformationDTO.campaignList', 
    'campaignRelation.terminationPreTermination.campaignInformationDTO.campaignList', 
    'campaignRelation.terminationConcurrentTermination.campaignInformationDTO.campaignList', 
    'campaignRelation.signupConcurrentSignup.campaignInformationDTO.campaignList',
    "topupInfo|smsSubtractionRate",
    "topupInfo|mmsSubtractionRate",
    "topupInfo|lmsSubtractionRate",
    "topupInfo|voiceCallSubtractionRate|onNetCall",
    "topupInfo|voiceCallSubtractionRate|offNetCall",
    "topupInfo|voiceCallSubtractionRate|landLineCall",
    "topupInfo|voiceCallSubtractionRate|valueAddedCall",
    "topupInfo|voiceCallSubtractionRate|videoCall",
    "topupInfo|dataSubtractionRate",
    'autoProductChange.changeRule.dateBase.value', 
    'autoProductChange.changeRule.date.value', 
    'autoProductChange.productNameAfterChange.pmProductId', 
    'autoProductChange.productNameAfterChange.legacyProductId', 
    'autoProductChange.productNameAfterChange.productName',
    'processInfo.onboardingDevice.deviceType.eligibility', 
    'processInfo.onboardingDevice.deviceType.valueList', 
    'processInfo.onboardingServiceCode.eligibility', 
    'processInfo.onboardingServiceCode.valueList',
    "processInfo|processMobileOnboardingType|onboardingTypeEligibility",
    "processInfo|processMobileOnboardingType|onboardingTypeEligibilityReason",
    "processInfo|onboardingChannel|channelConfigType",
    "processInfo|onboardingChannel|partnerCode",
    "processInfo|onboardingChannel|channelType",
    "processInfo|onboardingChannel|channelMiddleCategory",
    "processInfo|onboardingChannel|channelMajorCategory",
    "processInfo|onboardingDevice|ruleTypes",
    "processInfo|onboardingDevice|modelName",
    "processInfo|onboardingDevice|modelGroupName",
    "processInfo|onboardingDevice|deviceType",
    "processInfo|onboardingDevice|purchaseRequiredDevice",
    "processInfo|onboardingServiceCode",
    "customerInfo|onboardingCustomer|businessCustomerSubtypeRule",
    "existingOnboardInfo|mobileOnboardType",
    "existingOnboardInfo|mobileOnboardType|onboardingTypeEligibility",
    "existingOnboardInfo|mobileOnboardType|onboardingTypeChangeReason",
    "existingOnboardInfo|onboardProcessDate|processDate",
    "existingOnboardInfo|onboardDateBase",
    "existingOnboardInfo|mobileOnboardChannel",
    "existingOnboardInfo|mobileOnboardChannel|channelConfigType",
    "existingOnboardInfo|mobileOnboardChannel|partnerCode",
    "existingOnboardInfo|mobileOnboardChannel|channelType",
    "existingOnboardInfo|mobileOnboardChannel|channelMiddleCategory",
    "existingOnboardInfo|mobileOnboardChannel|channelMajorCategory",
    "optionData|dataOptionProvidingMethod|detailedDataOptionId",
    "optionData|dataOptionProvidingMethod|legacyDataOptionCode",
    "optionData|dataOptionProvidingMethod|pmDataOptionCode"
]

In [251]:
drop_columns = []
for col_nm in meta_data.columns:
    for drop_col in drop_cols:
        if col_nm.lower().startswith(drop_col.lower()):
            drop_columns.append(col_nm)
            break

drop_columns

['managementInfo.productId.value']

In [252]:
try:
    meta_data.drop(columns=drop_columns, inplace=True)
except KeyError as e:
    print("존재하지 않는 컬럼을 무시하고 계속 진행합니다.")

# Valid Column write 

In [257]:
save_path = "valid_columns.txt"
valid_columns = pd.DataFrame([meta_data.columns])
valid_columns.to_csv(save_path, sep='\n', header=False, index=False, encoding='utf-8')

In [8]:
# column_list = list(column_mapper.keys())

In [9]:
# meta_column_mapper = {}
# fields = list(meta_data.columns)
# for field in fields:
#     field_text = field.split('.')
#     if field_text[-1].lower() in ['value', 'valuelist']:
#         field_text.pop()
#         corr_field = '.'.join(field_text)
#     else:
#         corr_field = field
#     meta_column_mapper[field] = corr_field

# meta_data = meta_data.rename(columns=meta_column_mapper)

In [10]:
# valid_column_set = set()
# invalid_column_list = []
# for field in column_list:
#     if  field not in meta_data.columns:
#         try:
#             field_split = field.split('.')
#             sub_field = '.'.join(field_split[:-1])
#             if sub_field in meta_data.columns:
#                 valid_column_set.add(sub_field)
#             else:
#                 invalid_column_list.append(field)
#         except:
#             pass
#     else:
#         valid_column_set.add(field)    

In [258]:
# valid_column_set.add("productBenefitConditions.allOfferBenefits")
# valid_column_list = list(valid_column_set)
# valid_meta = meta_data[valid_column_list]
# import json

#JSON 파일에서 column_mapper 로드
# with open('column_mapper.json', 'r') as json_file:
#    column_dict = json.load(json_file)

# Columnr값 요약어로 줄이기
* 처리 방식: 너무 길어서 -> 조금 짧은 값으로 줄임 

In [266]:
token_set  = set()
dup_set = set()

for field in meta_data.columns:
    field = field.replace('|', '.')
    field_parts = field.split('.')
    if field_parts[-1] in token_set:
        dup_set.add(field_parts[-1])
    token_set.add(field_parts[-1])

column_mapper = {}

for field in meta_data.columns:
    field = field.replace('|', '.')
    field_parts = field.split('.')
    top_filed = field_parts[0]

    if field_parts[-1] == 'value' or field_parts[-1] == 'valueList':
        if field_parts[-2] in dup_set:
            value = top_filed + '.' + '.'.join(field_parts[-2:-1])
            column_mapper[field] = value
        else:
            if len(field_parts) > 1:
                value = top_filed + '.' + field_parts[-2]
                column_mapper[field] = value
            else:
                value = field_parts[-2]
                column_mapper[field] = value
    else:
        if field_parts[-1] in dup_set:
            value = top_filed + '.' + '.'.join(field_parts[-2:])
            column_mapper[field] = value
        else:
            if len(field_parts) > 1:
                value = top_filed + '.' + field_parts[-1]
                column_mapper[field] = value
            else:
                value = field_parts[-1]
                column_mapper[field] = value


# column mapper[dict] -> json으로 저장      

filename = 'column_mapper.json'

if not os.path.exists(filename):
    with open(filename, 'w') as json_file:
        json.dump(column_mapper, json_file, indent=4)
    print(f"파일 '{filename}'이 저장되었습니다.")
else:
    print(f"파일 '{filename}'이 이미 존재합니다.")

파일 'column_mapper.json'이 이미 존재합니다.


In [265]:
not_matched_columns = []
for column in meta_data.columns:
    try:
        column_mapper[column]
    except KeyError as e:
        not_matched_columns.append(column)

not_matched_columns

[]

In [378]:
# renamed_valid_meta = meta_data.rename(columns=column_mapper)
renamed_valid_meta = meta_data.copy()

In [379]:
renamed_valid_meta.head(3)

,pmProductID,voice.includedVoiceCall.value,voice.includedVideoOrValueAddedCall.value,voice.includedVoiceCallVideoOrValueAddedCallSeparateSetting.voiceCallRange,voice.includedVoiceCallTospecifiedNumbers.voiceCallRange,smsText.includedText.value,smsText.includedTextSeparateSetting.textRange,includedData.value,additionalDataUsage.includedDataForSharingAndTethering.value,additionalDataUsage.includedMVoIP.value,additionalDataUsage.includedDataSeparateSetting.dataRange,dataQoS.appliedSpeed.value,seniorDataExceedLimit.availableToApply.value,generalDataExceedLimit.availableToApply.value,monthlyPrice.monthlyPrice.value,monthlyPrice.monthlyPriceWithoutVAT.value,monthlyPrice.monthlyPriceWithSelectableInstallment.value,monthlyPrice.billingMethod.value,optionData.dataOptionProvidingMethod,deductibleInfo.deductibilityForDisability.value,benefitOfVoiceCall.performRefill.voiceCallRefillRange,benefitOfData.dataOptionRefill.dataRefillAmount.value,benefitOfData.dataOptionRefill.dataRefillCouponGiftingAvailability.value,benefitOfData.dataOptionGift.maximumShareAmount.value,benefitOfData.dataOptionGiftReceiving.dataGiftReceivingAvailability.value,managementInfo.statusOfOperation.value,managementInfo.classifiedGroup.value,managementInfo.productName.value,managementInfo.productNameInEnglish.value,managementInfo.lineup.value,managementInfo.marketingKeyword.valueList,managementInfo.generation.valueList,managementInfo.mappedProductCode.productCode.valueList,managementInfo.productDescription.value,managementInfo.productSubscriptionCondition.value,managementInfo.productSubscriptionMethod.value,salesInfo.netPrice.value,topupInfo.reChargeAvailability.availability.value,customerInfo.onboardingCustomer.ageRule,otherOnboardInfo.productChangeLineup.availability.value,otherOnboardInfo.directPlanOnboard.value,otherOnboardInfo.fixedPlanContractConcurrentSignupRestriction.value,otherOnboardInfo.tsupportFundOnboard.value,productRelation.signupPreTermination.productInformation.productList,productRelation.signupPreTermination.productInformation.groupList,productRelation.signupConcurrentTermination.productInformation.productList,productRelation.signupConcurrentTermination.productInformation.groupList,productRelation.terminationPreTermination.productInformation.productList,productRelation.terminationPreTermination.productInformation.groupList,productRelation.terminationConcurrentTermination.productInformation.productList,productRelation.terminationConcurrentTermination.productInformation.groupList,productRelation.subRule.productInformation.productList,productRelation.subRule.productInformation.groupList,commonRule.planChangeCondition.dailyPlanChangeLimit.value,commonRule.planChangeCondition.monthlyPlanChangeLimit.value,productBenefitConditions.allOfferBenefits,productBenefitConditions.optionalOfferBenefits.optionalOfferBenefitDetailList,productBenefitConditions.optionalOfferBenefits.mutuallyExclusiveBenefits,optionData.optionDataName.value,optionData.selectionMethod.value,optionData.totalNumOfOptions.value,optionData.minNumOfOptionSelectable.value,optionData.maxNumOfOptionSelectable.value,deductibleInfo.additionalOfferForDisabilities.value,productBenefitConditions.optionalOfferBenefits.selectableBenefitCount.value,productBenefitConditions.optionalOfferBenefits.selectableBenefitCountPeriodFrom.value,productBenefitConditions.optionalOfferBenefits.selectableBenefitCountPeriodTo.value,productBenefitConditions.optionalOfferBenefits.isAutoEnrollmentBenefitOnSignup.value,customerInfo.onboardingCustomer.customerTypeRule.eligibility,customerInfo.onboardingCustomer.customerTypeRule.valueList,customerInfo.onboardingCustomer.businessCustomerSubtypeRule.eligibility,customerInfo.onboardingCustomer.businessCustomerSubtypeRule.valueList,otherOnboardInfo.duplicateNameOnboard.domain.valueList,otherOnboardInfo.duplicateNameOnboard.productGroup.productInformation.groupList,productBenefitConditions.optionalOfferBenefits.autoSelectedBenefit.productInformation.productList,topupInfo.chargeAmo

# 전처리 내용 
* 관련 필드 카테고리 : 비 관계성 데이터 
* 필드명: 
    - voice.includedVoiceCallVideoOrValueAddedCallSeparateSetting.voiceCallRange, 
    - voice.includedVoiceCallTospecifiedNumbers.voiceCallRange, 
    - benefitOfVoiceCall.performRefill.voiceCallRefillRange
* 처리내용: voiceCallRange -> providingAmount와 Range로 분리 
* 위 처리 이유: pd.json_normalize로 읽으면 voiceCallRange까지만 계위가 내려옴

In [380]:
#renamed_valid_meta.iloc[28]["benefitOfVoiceCall.performRefill.voiceCallRefillRange"]
def extract_refill_amount(data):
    """
    benefitOfVoiceCall.performRefill.voiceCallRefillRange 데이터에서 refillAmount 값을 추출
    데이터 예시: [
        {
            'range': {'valueList': ['망내통화', '망외통화', '부가통화', '유선통화']},
            'refillAmount': {'value': '20%'}
        }
    ]
    """
    if isinstance(data, str):
        try:
            # 문자열인 경우 리스트로 변환
            data = ast.literal_eval(data)
        except:
            return None
    
    if isinstance(data, list) and len(data) > 0:
        # 리스트의 첫 번째 항목에서 refillAmount 추출
        first_item = data[0]
        if isinstance(first_item, dict) and 'refillAmount' in first_item:
            refill_amount_obj = first_item['refillAmount']
            if isinstance(refill_amount_obj, dict) and 'value' in refill_amount_obj:
                return refill_amount_obj['value']
    
    return None

def extract_range_value_list(data):
    """
    benefitOfVoiceCall.performRefill.voiceCallRefillRange 데이터에서 range의 valueList를 추출
    데이터 예시: [
        {
            'range': {'valueList': ['망내통화', '망외통화', '부가통화', '유선통화']},
            'refillAmount': {'value': '20%'}
        }
    ]
    """
    if isinstance(data, str):
        try:
            # 문자열인 경우 리스트로 변환
            data = ast.literal_eval(data)
        except:
            return None
    
    if isinstance(data, list) and len(data) > 0:
        # 리스트의 첫 번째 항목에서 range의 valueList 추출
        first_item = data[0]
        if isinstance(first_item, dict) and 'range' in first_item:
            range_obj = first_item['range']
            if isinstance(range_obj, dict) and 'valueList' in range_obj:
                return range_obj['valueList']
    
    return None

In [381]:
renamed_valid_meta['benefitOfVoiceCall.performRefill.voiceCallRefillRange.refillAmount'] = renamed_valid_meta['benefitOfVoiceCall.performRefill.voiceCallRefillRange'].apply(extract_refill_amount)
renamed_valid_meta['benefitOfVoiceCall.performRefill.voiceCallRefillRange.range'] = renamed_valid_meta['benefitOfVoiceCall.performRefill.voiceCallRefillRange'].apply(extract_range_value_list)

In [382]:
# 전처리끝나면 drop
renamed_valid_meta.drop(columns="benefitOfVoiceCall.performRefill.voiceCallRefillRange", inplace=True)

In [383]:
def extract_providing_amount(data):
    """
    voice.includedVoiceCallVideoOrValueAddedCallSeparateSetting.voiceCallRange 데이터에서 providingAmount 값을 추출
    데이터 예시: [
        {
            'range': {'valueList': ['망내통화', '망외통화', '부가통화', '유선통화']},
            'providingAmount': {'value': '60분'}
        }
    ]
    """
    if isinstance(data, str):
        try:
            # 문자열인 경우 리스트로 변환
            data = ast.literal_eval(data)
        except:
            return None
    
    if isinstance(data, list) and len(data) > 0:
        # 리스트의 첫 번째 항목에서 refillAmount 추출
        first_item = data[0]
        if isinstance(first_item, dict) and 'providingAmount' in first_item:
            refill_amount_obj = first_item['providingAmount']
            if isinstance(refill_amount_obj, dict) and 'value' in refill_amount_obj:
                return refill_amount_obj['value']
    
    return None

def extract_range_value_list(data):
    """
    renamed_valid_meta["voice.includedVoiceCallVideoOrValueAddedCallSeparateSetting.voiceCallRange"].iloc[28]
    voice.includedVoiceCallVideoOrValueAddedCallSeparateSetting.voiceCallRange 데이터에서 providingAmount 값을 추출
    데이터 예시: [
        {
            'range': {'valueList': ['망내통화', '망외통화', '부가통화', '유선통화']},
            'providingAmount': {'value': '60분'}
        }
    ]
    """
    if isinstance(data, str):
        try:
            # 문자열인 경우 리스트로 변환
            data = ast.literal_eval(data)
        except:
            return None
    
    if isinstance(data, list) and len(data) > 0:
        # 리스트의 첫 번째 항목에서 range의 valueList 추출
        first_item = data[0]
        if isinstance(first_item, dict) and 'range' in first_item:
            range_obj = first_item['range']
            if isinstance(range_obj, dict) and 'valueList' in range_obj:
                return range_obj['valueList']
    
    return None

In [384]:
renamed_valid_meta['voice.includedVoiceCallVideoOrValueAddedCallSeparateSetting.voiceCallRange.providingAmount'] = renamed_valid_meta['voice.includedVoiceCallVideoOrValueAddedCallSeparateSetting.voiceCallRange'].apply(extract_providing_amount)
renamed_valid_meta['voice.includedVoiceCallVideoOrValueAddedCallSeparateSetting.voiceCallRange.range'] = renamed_valid_meta['voice.includedVoiceCallVideoOrValueAddedCallSeparateSetting.voiceCallRange'].apply(extract_range_value_list)


In [385]:
renamed_valid_meta.drop(columns="voice.includedVoiceCallVideoOrValueAddedCallSeparateSetting.voiceCallRange", inplace=True)

In [386]:
def extract_providing_amount(data):
    """
    voice.includedVoiceCallTospecifiedNumbers.voiceCallRange 데이터에서 providingAmount 값을 추출
    데이터 예시: [
        [{'range': {'valueList': ['망내통화']},
        'providingAmount': {'value': '무제한'},
        'numberOfService': {'value': '2'}}]
    """
    if isinstance(data, str):
        try:
            # 문자열인 경우 리스트로 변환
            data = ast.literal_eval(data)
        except:
            return None
    
    if isinstance(data, list) and len(data) > 0:
        # 리스트의 첫 번째 항목에서 refillAmount 추출
        first_item = data[0]
        if isinstance(first_item, dict) and 'providingAmount' in first_item:
            refill_amount_obj = first_item['providingAmount']
            if isinstance(refill_amount_obj, dict) and 'value' in refill_amount_obj:
                return refill_amount_obj['value']
    
    return None

def extract_range_value_list(data):
    """
    voice.includedVoiceCallTospecifiedNumbers.voiceCallRange 데이터에서 providingAmount 값을 추출
    데이터 예시: [
        [{'range': {'valueList': ['망내통화']},
        'providingAmount': {'value': '무제한'},
        'numberOfService': {'value': '2'}}]
    """
    if isinstance(data, str):
        try:
            # 문자열인 경우 리스트로 변환
            data = ast.literal_eval(data)
        except:
            return None
    
    if isinstance(data, list) and len(data) > 0:
        # 리스트의 첫 번째 항목에서 range의 valueList 추출
        first_item = data[0]
        if isinstance(first_item, dict) and 'range' in first_item:
            range_obj = first_item['range']
            if isinstance(range_obj, dict) and 'valueList' in range_obj:
                return range_obj['valueList']
    
    return None

def extract_numberOfService(data):
    """
        voice.includedVoiceCallTospecifiedNumbers.voiceCallRange 데이터에서 providingAmount 값을 추출
        데이터 예시: [
            [{'range': {'valueList': ['망내통화']},
            'providingAmount': {'value': '무제한'},
            'numberOfService': {'value': '2'}}]
    """
    if isinstance(data, str):
        try:
            # 문자열인 경우 리스트로 변환
            data = ast.literal_eval(data)
        except:
            return None
    
    if isinstance(data, list) and len(data) > 0:
        # 리스트의 첫 번째 항목에서 range의 valueList 추출
        first_item = data[0]
        if isinstance(first_item, dict) and 'numberOfService' in first_item:
            range_obj = first_item['numberOfService']
            if isinstance(range_obj, dict) and 'value' in range_obj:
                return range_obj['value']
    
    return None

In [387]:
renamed_valid_meta['voice.includedVoiceCallTospecifiedNumbers.voiceCallRange.providingAmount'] = renamed_valid_meta['voice.includedVoiceCallTospecifiedNumbers.voiceCallRange'].apply(extract_providing_amount)
renamed_valid_meta['voice.includedVoiceCallTospecifiedNumbers.voiceCallRange.range'] = renamed_valid_meta['voice.includedVoiceCallTospecifiedNumbers.voiceCallRange'].apply(extract_range_value_list)
renamed_valid_meta['voice.includedVoiceCallTospecifiedNumbers.voiceCallRange.numberOfService'] = renamed_valid_meta['voice.includedVoiceCallTospecifiedNumbers.voiceCallRange'].apply(extract_numberOfService)


In [388]:
renamed_valid_meta.drop(columns="voice.includedVoiceCallTospecifiedNumbers.voiceCallRange", inplace=True)

In [389]:
renamed_valid_meta.head(3)

,pmProductID,voice.includedVoiceCall.value,voice.includedVideoOrValueAddedCall.value,smsText.includedText.value,smsText.includedTextSeparateSetting.textRange,includedData.value,additionalDataUsage.includedDataForSharingAndTethering.value,additionalDataUsage.includedMVoIP.value,additionalDataUsage.includedDataSeparateSetting.dataRange,dataQoS.appliedSpeed.value,seniorDataExceedLimit.availableToApply.value,generalDataExceedLimit.availableToApply.value,monthlyPrice.monthlyPrice.value,monthlyPrice.monthlyPriceWithoutVAT.value,monthlyPrice.monthlyPriceWithSelectableInstallment.value,monthlyPrice.billingMethod.value,optionData.dataOptionProvidingMethod,deductibleInfo.deductibilityForDisability.value,benefitOfData.dataOptionRefill.dataRefillAmount.value,benefitOfData.dataOptionRefill.dataRefillCouponGiftingAvailability.value,benefitOfData.dataOptionGift.maximumShareAmount.value,benefitOfData.dataOptionGiftReceiving.dataGiftReceivingAvailability.value,managementInfo.statusOfOperation.value,managementInfo.classifiedGroup.value,managementInfo.productName.value,managementInfo.productNameInEnglish.value,managementInfo.lineup.value,managementInfo.marketingKeyword.valueList,managementInfo.generation.valueList,managementInfo.mappedProductCode.productCode.valueList,managementInfo.productDescription.value,managementInfo.productSubscriptionCondition.value,managementInfo.productSubscriptionMethod.value,salesInfo.netPrice.value,topupInfo.reChargeAvailability.availability.value,customerInfo.onboardingCustomer.ageRule,otherOnboardInfo.productChangeLineup.availability.value,otherOnboardInfo.directPlanOnboard.value,otherOnboardInfo.fixedPlanContractConcurrentSignupRestriction.value,otherOnboardInfo.tsupportFundOnboard.value,productRelation.signupPreTermination.productInformation.productList,productRelation.signupPreTermination.productInformation.groupList,productRelation.signupConcurrentTermination.productInformation.productList,productRelation.signupConcurrentTermination.productInformation.groupList,productRelation.terminationPreTermination.productInformation.productList,productRelation.terminationPreTermination.productInformation.groupList,productRelation.terminationConcurrentTermination.productInformation.productList,productRelation.terminationConcurrentTermination.productInformation.groupList,productRelation.subRule.productInformation.productList,productRelation.subRule.productInformation.groupList,commonRule.planChangeCondition.dailyPlanChangeLimit.value,commonRule.planChangeCondition.monthlyPlanChangeLimit.value,productBenefitConditions.allOfferBenefits,productBenefitConditions.optionalOfferBenefits.optionalOfferBenefitDetailList,productBenefitConditions.optionalOfferBenefits.mutuallyExclusiveBenefits,optionData.optionDataName.value,optionData.selectionMethod.value,optionData.totalNumOfOptions.value,optionData.minNumOfOptionSelectable.value,optionData.maxNumOfOptionSelectable.value,deductibleInfo.additionalOfferForDisabilities.value,productBenefitConditions.optionalOfferBenefits.selectableBenefitCount.value,productBenefitConditions.optionalOfferBenefits.selectableBenefitCountPeriodFrom.value,productBenefitConditions.optionalOfferBenefits.selectableBenefitCountPeriodTo.value,productBenefitConditions.optionalOfferBenefits.isAutoEnrollmentBenefitOnSignup.value,customerInfo.onboardingCustomer.customerTypeRule.eligibility,customerInfo.onboardingCustomer.customerTypeRule.valueList,customerInfo.onboardingCustomer.businessCustomerSubtypeRule.eligibility,customerInfo.onboardingCustomer.businessCustomerSubtypeRule.valueList,otherOnboardInfo.duplicateNameOnboard.domain.valueList,otherOnboardInfo.duplicateNameOnboard.productGroup.productInformation.groupList,productBenefitConditions.optionalOfferBenefits.autoSelectedBenefit.productInformation.productList,topupInfo.chargeAmount.minimumChargeAmount.value,topupInfo.chargeAmount.maximumChargeAmount.value,otherOnboardInfo.specialCustomerOnboard.isSoldier.value,otherOnboardInfo.duplicateNameOnboard.productGroup.

# 전처리 내용 
* 관련 필드 카테고리 : 비 관계성 데이터 
* 필드명: 필드 병합: (includeVoiceCall 관련 필드)
* 처리내용: 필드 병합 

In [390]:
def consolidate_voice_fields(df):
    """
    음성통화 관련 필드들을 통합하는 함수
    """
    # 복사본 생성
    result_df = df.copy()
    
    # voice.includedVoiceCall 통합
    # "voice.includedVoiceCall" 또는 "voice.includedVoiceCallTospecifiedNumbers.voiceCallRange.providingAmount" 중 
    # 값이 NaN 혹은 None이 아닌 값이 존재하면 voice.includedVoiceCall로 할당
    def consolidate_voice_call(row):
        voice_call = row.get('voice.includedVoiceCall.value')
        voice_specified = row.get('voice.includedVoiceCallTospecifiedNumbers.voiceCallRange.providingAmount')
        
        # voice.includedVoiceCall이 유효한 값이면 그대로 사용
        if pd.notna(voice_call) and voice_call is not None and str(voice_call).strip() != '':
            return voice_call
        # 그렇지 않으면 voice.includedVoiceCallTospecifiedNumbers 값 사용
        elif pd.notna(voice_specified) and voice_specified is not None and str(voice_specified).strip() != '':
            return voice_specified
        else:
            return voice_call  # 둘 다 없으면 원래 값 유지
    
    # voice.includedVideoOrValueAddedCall 통합
    # "voice.includedVideoOrValueAddedCall" 또는 "voice.includedVoiceCallVideoOrValueAddedCallSeparateSetting.voiceCallRange.providingAmount" 중
    # 값이 존재하는 값으로 voice.includedVideoOrValueAddedCall로 할당
    def consolidate_video_call(row):
        video_call = row.get('voice.includedVideoOrValueAddedCall.value')
        video_separate = row.get('voice.includedVoiceCallVideoOrValueAddedCallSeparateSetting.voiceCallRange.providingAmount')
        
        # voice.includedVideoOrValueAddedCall이 유효한 값이면 그대로 사용
        if pd.notna(video_call) and video_call is not None and str(video_call).strip() != '':
            return video_call
        # 그렇지 않으면 separate setting 값 사용
        elif pd.notna(video_separate) and video_separate is not None and str(video_separate).strip() != '':
            return video_separate
        else:
            return video_call  # 둘 다 없으면 원래 값 유지
    
    # 통합 적용
    result_df['voice.includedVoiceCall.value'] = result_df.apply(consolidate_voice_call, axis=1)
    result_df['voice.includedVideoOrValueAddedCall.value'] = result_df.apply(consolidate_video_call, axis=1)
    
    return result_df

In [391]:
renamed_valid_meta = consolidate_voice_fields(renamed_valid_meta)

In [411]:
# 병합 끝난 후 제거 
drop_cols = ["voice.includedVoiceCallTospecifiedNumbers.voiceCallRange.providingAmount", "voice.includedVoiceCallVideoOrValueAddedCallSeparateSetting.voiceCallRange.providingAmount"]
renamed_valid_meta.drop(columns=drop_cols, inplace=True)



# 벙합 특이 케이스 
* PA00000228	
    - voice.includedVoiceCall.value: 대표 회선의 음성통화 제공량 이용	
    - voice.includedVideoOrValueAddedCall.value: 대표 회선의 음성통화 제공량 이용	

In [392]:
renamed_valid_meta[["pmProductID","voice.includedVoiceCall.value", "voice.includedVideoOrValueAddedCall.value" ,"voice.includedVoiceCallTospecifiedNumbers.voiceCallRange.providingAmount", "voice.includedVoiceCallVideoOrValueAddedCallSeparateSetting.voiceCallRange.providingAmount"]].head(3)

,pmProductID,voice.includedVoiceCall.value,voice.includedVideoOrValueAddedCall.value,voice.includedVoiceCallTospecifiedNumbers.voiceCallRange.providingAmount,voice.includedVoiceCallVideoOrValueAddedCallSeparateSetting.voiceCallRange.providingAmount
0,PA00000001,무제한,300분,None,None
1,PA00000002,무제한,100분,None,None
2,PA00000003,무제한,300분,None,None


# 전처리 내용 
* 관련 필드 카테고리 : 관계성 데이터
* 필드명: productBenefitConditions.allOfferBenefits, productBenefitConditions.optionalOfferBenefits.optionalOfferBenefitDetailList 
* 처리내용: Id, name 만 남기고 전부 제거

In [393]:
productBenefit_df = renamed_valid_meta[["pmProductID", "productBenefitConditions.allOfferBenefits"]]

In [394]:
from typing import Union
import ast

def extract_val(data, keys: Union[str, list]):
    """재귀적으로 중첩된 딕셔너리에서 값을 추출"""
    if data is None:
        return None
        
    if isinstance(keys, str):
        try:
            return data.get(keys) if isinstance(data, dict) else None
        except:
            return None
        
    elif isinstance(keys, list) and len(keys) > 0:
        if len(keys) == 1:
            return data.get(keys[0]) if isinstance(data, dict) else None
        else:
            # 첫 번째 키로 값을 가져온 후 나머지 키들로 재귀 호출
            first_key = keys[0]
            remaining_keys = keys[1:]
            next_data = data.get(first_key) if isinstance(data, dict) else None
            return extract_val(next_data, remaining_keys)
    else:
        return None


def parse_allOffer(data):
    """
        [
            {
            'benefitInfo': 
                {
                'productInformation': {
                    'productList': [
                        {
                            'pmProductId': 'BA00000047',
                            'productName': 'Wavve 2천원 할인'
                        }
                        ]   
                    }
                },
            'benefitPeriodTo': {'value': '9999.12.31'},
            'benefitAutoEnrollment': {'isBenefitAutoEnrollment': {'value': 'Y'},
            'benefitAutoEnrollmentReferenceTime': {'value': '혜택 적용 상품 가입시'}},
            'benefitAutoCancellation': {'isBenefitAutoCancellation': {'value': 'Y'},
            'benefitAutoTerminationReferenceTime': {'value': '혜택 적용 상품 해지시'}}}
        ]
    """

    def parse(list_data: list) -> list:
        """ 초기 버젼에서는 product_id, product_name 만 추출"""
        result_data = [] 
        for item in list_data:
            benefitPeriodTo = extract_val(item, keys=['benefitPeriodTo'])
            benefitAutoEnrollment = extract_val(item, keys=['benefitAutoEnrollment'])
            benefitAutoEnrollmentReferenceTime = extract_val(item, keys=['benefitAutoEnrollmentReferenceTime'])
            benefitAutoCancellation = extract_val(item, keys=['benefitAutoCancellation'])
            benefitAutoTerminationReferenceTime = extract_val(item, keys=['benefitAutoTerminationReferenceTime'])
            product_list = extract_val(item, keys=['benefitInfo', 'productInformation', 'productList'])
            if product_list:
                for product in product_list:
                    product_id = product['pmProductId']
                    product_name = product['productName']
                    result_data.append(
                        {
                            "productId": product_id,
                            "productName": product_name,
                        }
                    )
        if not result_data:
            return None
        
        return result_data

    if isinstance(data, dict):
        data = parse([data])

    elif isinstance(data, list):
        data = parse(data)
    else:
        try:
            data = ast.literal_eval(data)
            data = parse(data)
        except:
            return None

    return data

# 디버깅을 위한 테스트 함수
def debug_parse_allOffer(data, index=None):
    """디버깅용 함수"""
    print(f"디버깅 - 인덱스: {index}")
    print(f"데이터 타입: {type(data)}")
    print(f"데이터 내용: {data}")
    
    result = parse_allOffer(data)
    print(f"파싱 결과: {result}")
    print("-" * 50)
    return result

# 적용
#productBenefit_df.loc[:, 'productBenefitConditions.allOfferBenefits.benefitInfo'] = productBenefit_df['productBenefitConditions.allOfferBenefits'].apply(parse_allOffer)

#디버깅이 필요한 경우 아래 코드 사용
for idx in range(min(5, len(productBenefit_df))):  # 처음 5개 행만 디버깅
    debug_parse_allOffer(productBenefit_df.iloc[idx]['productBenefitConditions.allOfferBenefits'], idx)

디버깅 - 인덱스: 0
데이터 타입: <class 'list'>
데이터 내용: [{'benefitInfo': {'productInformation': {'productList': [{'pmProductId': 'BA00000047', 'productName': 'Wavve 2천원 할인'}]}}, 'benefitPeriodTo': {'value': '9999.12.31'}, 'benefitAutoEnrollment': {'isBenefitAutoEnrollment': {'value': 'Y'}, 'benefitAutoEnrollmentReferenceTime': {'value': '혜택 적용 상품 가입시'}}, 'benefitAutoCancellation': {'isBenefitAutoCancellation': {'value': 'Y'}, 'benefitAutoTerminationReferenceTime': {'value': '혜택 적용 상품 해지시'}}}]
파싱 결과: [{'productId': 'BA00000047', 'productName': 'Wavve 2천원 할인'}]
--------------------------------------------------
디버깅 - 인덱스: 1
데이터 타입: <class 'float'>
데이터 내용: nan
파싱 결과: None
--------------------------------------------------
디버깅 - 인덱스: 2
데이터 타입: <class 'float'>
데이터 내용: nan
파싱 결과: None
--------------------------------------------------
디버깅 - 인덱스: 3
데이터 타입: <class 'float'>
데이터 내용: nan
파싱 결과: None
--------------------------------------------------
디버깅 - 인덱스: 4
데이터 타입: <class 'list'>
데이터 내용: [{'benefitInfo'

In [395]:
productBenefit_df.loc[:, 'productBenefitConditions.allOfferBenefits.benefitInfo'] = productBenefit_df['productBenefitConditions.allOfferBenefits'].apply(parse_allOffer)


/var/folders/hc/v5sxz14d4cd81kh7q7y7grh81lcq7q/T/ipykernel_39278/3336404138.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  productBenefit_df.loc[:, 'productBenefitConditions.allOfferBenefits.benefitInfo'] = productBenefit_df['productBenefitConditions.allOfferBenefits'].apply(parse_allOffer)


In [362]:
productBenefit_df[["pmProductID", "productBenefitConditions.allOfferBenefits.benefitInfo"]].to_csv("productBenefit_df.csv", index=False, encoding="utf-8")

In [396]:
productOptionalBenefit_df = renamed_valid_meta[["pmProductID", "productBenefitConditions.optionalOfferBenefits.optionalOfferBenefitDetailList"]]

In [397]:
def extract_val(data, keys: Union[str, list]):
    """재귀적으로 중첩된 딕셔너리에서 값을 추출"""
    if data is None:
        return None
        
    if isinstance(keys, str):
        try:
            return data.get(keys) if isinstance(data, dict) else None
        except:
            return None
        
    elif isinstance(keys, list) and len(keys) > 0:
        if len(keys) == 1:
            return data.get(keys[0]) if isinstance(data, dict) else None
        else:
            # 첫 번째 키로 값을 가져온 후 나머지 키들로 재귀 호출
            first_key = keys[0]
            remaining_keys = keys[1:]
            next_data = data.get(first_key) if isinstance(data, dict) else None
            return extract_val(next_data, remaining_keys)
    else:
        return None

def parse_Offer(data):
    """
    productBenefitConditions.optionalOfferBenefits.optionalOfferBenefitDetailList 데이터를 파싱하여 pmProductId, productName정보를 추출
    sampleData: productOptionalBenefit_df.iloc[9]["productBenefitConditions.optionalOfferBenefits.optionalOfferBenefitDetailList"][0]
    {   'benefitInfo': {'productInformation': {'productList': [{'pmProductId': 'BA00000038',
        'productName': 'FLO 무료'}]}},
        'benefitPeriodTo': {'value': '9999.12.31'},
        'benefitChangeReferenceDate': {'value': '직전 혜택 변경일'},
        'benefitChangeAvailableDate': {'value': '즉시'}
    }
    """

    def parse(list_data: list) -> list:
        """ 초기 버젼에서는 product_id, product_name 만 추출"""
        result_data = [] 
        for item in list_data:
            product_list = extract_val(item, keys=['benefitInfo', 'productInformation', 'productList'])
            if product_list:
                for product in product_list:
                    product_id = product['pmProductId']
                    product_name = product['productName']
                    result_data.append(
                        {
                            "productId": product_id,
                            "productName": product_name,
                        }
                    )
        if not result_data:
            return None
        
        return result_data

    if isinstance(data, dict):
        data = parse([data])

    elif isinstance(data, list):
        data = parse(data)
    else:
        try:
            data = ast.literal_eval(data)
            data = parse(data)
        except:
            return None

    return data


# 디버깅을 위한 테스트 함수
def debug_parse_allOffer(data, index=None):
    """디버깅용 함수"""
    print(f"디버깅 - 인덱스: {index}")
    print(f"데이터 타입: {type(data)}")
    print(f"데이터 내용: {data}")
    
    result = parse_allOffer(data)
    print(f"파싱 결과: {result}")
    print("-" * 50)
    return result

#디버깅이 필요한 경우 아래 코드 사용
for idx in range(min(20, len(productOptionalBenefit_df))):  # 처음 5개 행만 디버깅
    debug_parse_allOffer(productOptionalBenefit_df.iloc[idx]['productBenefitConditions.optionalOfferBenefits.optionalOfferBenefitDetailList'], idx)

디버깅 - 인덱스: 0
데이터 타입: <class 'list'>
데이터 내용: []
파싱 결과: None
--------------------------------------------------
디버깅 - 인덱스: 1
데이터 타입: <class 'list'>
데이터 내용: []
파싱 결과: None
--------------------------------------------------
디버깅 - 인덱스: 2
데이터 타입: <class 'list'>
데이터 내용: []
파싱 결과: None
--------------------------------------------------
디버깅 - 인덱스: 3
데이터 타입: <class 'list'>
데이터 내용: []
파싱 결과: None
--------------------------------------------------
디버깅 - 인덱스: 4
데이터 타입: <class 'list'>
데이터 내용: []
파싱 결과: None
--------------------------------------------------
디버깅 - 인덱스: 5
데이터 타입: <class 'list'>
데이터 내용: []
파싱 결과: None
--------------------------------------------------
디버깅 - 인덱스: 6
데이터 타입: <class 'list'>
데이터 내용: []
파싱 결과: None
--------------------------------------------------
디버깅 - 인덱스: 7
데이터 타입: <class 'list'>
데이터 내용: []
파싱 결과: None
--------------------------------------------------
디버깅 - 인덱스: 8
데이터 타입: <class 'list'>
데이터 내용: []
파싱 결과: None
--------------------------------------------------
디버깅 - 인덱스:

In [398]:
productOptionalBenefit_df.loc[:, 'productBenefitConditions.optionalOfferBenefits.optionalOfferBenefitDetailList'] = productOptionalBenefit_df['productBenefitConditions.optionalOfferBenefits.optionalOfferBenefitDetailList'].apply(parse_Offer)


In [399]:
productOptionalBenefit_df[["pmProductID", "productBenefitConditions.optionalOfferBenefits.optionalOfferBenefitDetailList"]].to_csv(f"productOptionalBenefit_df.csv", index=False, encoding="utf-8")

In [424]:
print(1)

1


# opetionData 처리

In [428]:
option_data_field = [
 'optionData.dataOptionProvidingMethod',
 'optionData.optionDataName.value', 
 'optionData.selectionMethod.value', 
 'optionData.totalNumOfOptions.value',
 'optionData.minNumOfOptionSelectable.value',
 'optionData.maxNumOfOptionSelectable.value'
]

optionData = renamed_valid_meta[['pmProductID'] + option_data_field]

In [367]:
optionData = renamed_valid_meta[["pmProductID", "optionData.dataOptionProvidingMethod"]]

In [437]:
#optionData.iloc[58]["optionData.dataOptionProvidingMethod"]

In [ ]:
def extract_val(data, keys: Union[str, list]):
    """재귀적으로 중첩된 딕셔너리에서 값을 추출"""
    if data is None:
        return None
        
    if isinstance(keys, str):
        try:
            return data.get(keys) if isinstance(data, dict) else None
        except:
            return None
        
    elif isinstance(keys, list) and len(keys) > 0:
        if len(keys) == 1:
            return data.get(keys[0]) if isinstance(data, dict) else None
        else:
            # 첫 번째 키로 값을 가져온 후 나머지 키들로 재귀 호출
            first_key = keys[0]
            remaining_keys = keys[1:]
            next_data = data.get(first_key) if isinstance(data, dict) else None
            return extract_val(next_data, remaining_keys)
    else:
        return None

def parse_field(data):
    """
    optionData.iloc[1]["optionData.dataOptionProvidingMethod"] 파싱하여 detailedDataOptionName, legacyDataOptionCode
    sampleData: optionData.iloc[1]["optionData.dataOptionProvidingMethod"]
    [
        {
            'detailedDataOptionName': {'value': '베이직 플러스 데이터 충전 75GB'},
            'providingDataOptionMethod': {'value': '데이터 추가'},
            'legacyDataOptionCode': {'value': 'NA00008243'},
            'dataOptionProvidingRecord': {'dataOptionExtraData': {'dataAmount': {'value': '75GB'},
            'dataQoSAfterUsedUp': {'value': 'QOS 적용 없음'}},
            'frequencyOfApplyingDataOption': {'frequency': {'value': '매월'}},
            'timePeriodForApplyingDataOption': {'timePeriod': {'value': '적용 없음'}},
            'dataOptionApplyLocation': {'value': '전 지역'},
            'dataOptionMonthlyPrice': {'value': '9000원'}}
        },
    ]
    """

    def parse(list_data: list) -> list:
        """ 초기 버젼에서는 product_id, product_name 만 추출"""
        result_data = [] 
        for item in list_data:
            name = extract_val(item, keys=['detailedDataOptionName', 'value'])
            id = extract_val(item, keys=['legacyDataOptionCode', 'value'])
            dataOptionProvidingRecord = extract_val(item, keys=['dataOptionProvidingRecord', 'dataOptionExtraData', 'dataAmount', 'value'])
            dataSubtractionDiscountRate = extract_val(item, keys=['dataOptionProvidingRecord', 'dataSubtractionDiscountRate', 'rateDiscount', 'value'])
            monthlyPrice = extract_val(item, keys=['dataOptionProvidingRecord','dataOptionMonthlyPrice', 'value'])
            result_data.append(
                {
                    "productId": id,
                    "productName": name,
                    "extradataAmount": dataOptionProvidingRecord,
                    "monthlyPrice": monthlyPrice,
                    "DiscountRate": dataSubtractionDiscountRate
                }
            )
        if not result_data:
            return None
        
        return result_data

    if isinstance(data, dict):
        data = parse([data])

    elif isinstance(data, list):
        data = parse(data)
    else:
        try:
            data = ast.literal_eval(data)
            data = parse(data)
        except:
            return None

    return data


# 디버깅을 위한 테스트 함수
def debug_parse(data, index=None):
    """디버깅용 함수"""
    print(f"디버깅 - 인덱스: {index}")
    print(f"데이터 타입: {type(data)}")
    print(f"데이터 내용: {data}")
    
    result = parse_field(data)
    print(f"파싱 결과: {result}")
    print("-" * 50)
    return result

#디버깅이 필요한 경우 아래 코드 사용
for idx in range(min(60, len(optionData))):  # 처음 5개 행만 디버깅
    debug_parse(optionData.iloc[idx]['optionData.dataOptionProvidingMethod'], idx)

디버깅 - 인덱스: 0
데이터 타입: <class 'list'>
데이터 내용: [{'detailedDataOptionName': {}, 'providingDataOptionMethod': {}, 'detailedDataOptionId': {}, 'legacyDataOptionCode': {}, 'dataOptionProvidingRecord': {'dataOptionExtraData': {'dataAmount': {}, 'dataQoSAfterUsedUp': {}}, 'qosAfterIncludedDataConsumed': {'dataQoSAfterUsedUp': {}}, 'dataSubtractionDiscountRate': {'rateDiscount': {}}, 'frequencyOfApplyingDataOption': {'frequency': {}}, 'timePeriodForApplyingDataOption': {'timePeriod': {}}, 'dataOptionApplyLocation': {}, 'dataOptionMonthlyPrice': {}}}]
{'detailedDataOptionName': {}, 'providingDataOptionMethod': {}, 'detailedDataOptionId': {}, 'legacyDataOptionCode': {}, 'dataOptionProvidingRecord': {'dataOptionExtraData': {'dataAmount': {}, 'dataQoSAfterUsedUp': {}}, 'qosAfterIncludedDataConsumed': {'dataQoSAfterUsedUp': {}}, 'dataSubtractionDiscountRate': {'rateDiscount': {}}, 'frequencyOfApplyingDataOption': {'frequency': {}}, 'timePeriodForApplyingDataOption': {'timePeriod': {}}, 'dataOptionApp

In [ ]:
optionData.loc[:, 'optionData.dataOptionProvidingMethod'] = optionData['optionData.dataOptionProvidingMethod'].apply(parse_field)


In [449]:
optionData.to_csv("optionData.csv", index=False, encoding="utf-8")

In [451]:
# 제거 
renamed_valid_meta.drop(columns=option_data_field, inplace=True)

In [425]:
drop_list = []
name = "productBenefitConditions"
for col_nm in renamed_valid_meta.columns:
    if col_nm.lower().startswith(name.lower()):
        drop_list.append(col_nm)
drop_list.extend(["productBenefitConditions.allOfferBenefits"])
drop_list

['optionData.dataOptionProvidingMethod',
 'optionData.optionDataName.value',
 'optionData.selectionMethod.value',
 'optionData.totalNumOfOptions.value',
 'optionData.minNumOfOptionSelectable.value',
 'optionData.maxNumOfOptionSelectable.value',
 'productBenefitConditions.allOfferBenefits']

In [ ]:
renamed_valid_meta.drop(columns=drop_list, inplace=True)

In [458]:
#renamed_valid_meta.to_csv("meta_preprocessing.csv", index=False, encoding="utf-8")

# 정규식 전처리
* 분 처리
    - voice.includedVoiceCall.value	['무제한', '0분', '30분', '대표 회선의 음성통화 제공량 이용']
    - voice.includedVideoOrValueAddedCall.value	['300분', '100분', '150분', nan, '50분', '60분', '400분', '30분', '70분','대표 회선의 음성통화 제공량 이용']
    deductibleInfo.additionalOfferForDisabilities.value	[nan, '200분', '250분', '150분']
* 건처리 
    - smsText.includedText.value ['기본제공', '50건', '80건']
    
* 원 처리
* GB, MB (용량 관련)
    - includedData.value [xxGb, '400MB', 무제한]
    - additionalDataUsage.includedDataForSharingAndTethering.value [xxGb, '400MB', 무제한, nan]
    - additionalDataUsage.includedMVoIP.value [xxGb, '400MB', 무제한, nan]
    - benefitOfData.dataOptionRefill.dataRefillAmount.value	
    - benefitOfData.dataOptionGift.maximumShareAmount.value	
* 속도 관련
    - dataQoS.appliedSpeed.value: ['1Mbps', nan, '5Mbps', '400Kbps', '400kbps', '3Mbps']
* 가격 (원처리)
    - monthlyPrice.monthlyPrice.value (xxx원)
    - monthlyPrice.monthlyPriceWithoutVAT.value	(처리필요할지 고민 필요, 개인적으로는 Filter로 사용 x)
    - monthlyPrice.monthlyPriceWithSelectableInstallment.value	(처리필요할지 고민 필요, 개인적으로는 Filter로 사용 x)
    - salesInfo.netPrice.value	(처리필요할지 고민 필요, 개인적으로는 Filter로 사용 x)

* % 비율 
    - benefitOfVoiceCall.performRefill.voiceCallRefillRange.refillAmount ['20%', None]

In [460]:
renamed_valid_meta["benefitOfVoiceCall.performRefill.voiceCallRefillRange.refillAmount"].unique()

array(['20%', None], dtype=object)

In [452]:
renamed_valid_meta.head(30)

,pmProductID,voice.includedVoiceCall.value,voice.includedVideoOrValueAddedCall.value,smsText.includedText.value,smsText.includedTextSeparateSetting.textRange,includedData.value,additionalDataUsage.includedDataForSharingAndTethering.value,additionalDataUsage.includedMVoIP.value,additionalDataUsage.includedDataSeparateSetting.dataRange,dataQoS.appliedSpeed.value,seniorDataExceedLimit.availableToApply.value,generalDataExceedLimit.availableToApply.value,monthlyPrice.monthlyPrice.value,monthlyPrice.monthlyPriceWithoutVAT.value,monthlyPrice.monthlyPriceWithSelectableInstallment.value,monthlyPrice.billingMethod.value,deductibleInfo.deductibilityForDisability.value,benefitOfData.dataOptionRefill.dataRefillAmount.value,benefitOfData.dataOptionRefill.dataRefillCouponGiftingAvailability.value,benefitOfData.dataOptionGift.maximumShareAmount.value,benefitOfData.dataOptionGiftReceiving.dataGiftReceivingAvailability.value,managementInfo.statusOfOperation.value,managementInfo.classifiedGroup.value,managementInfo.productName.value,managementInfo.productNameInEnglish.value,managementInfo.lineup.value,managementInfo.marketingKeyword.valueList,managementInfo.generation.valueList,managementInfo.mappedProductCode.productCode.valueList,managementInfo.productDescription.value,managementInfo.productSubscriptionCondition.value,managementInfo.productSubscriptionMethod.value,salesInfo.netPrice.value,topupInfo.reChargeAvailability.availability.value,customerInfo.onboardingCustomer.ageRule,otherOnboardInfo.productChangeLineup.availability.value,otherOnboardInfo.directPlanOnboard.value,otherOnboardInfo.fixedPlanContractConcurrentSignupRestriction.value,otherOnboardInfo.tsupportFundOnboard.value,productRelation.signupPreTermination.productInformation.productList,productRelation.signupPreTermination.productInformation.groupList,productRelation.signupConcurrentTermination.productInformation.productList,productRelation.signupConcurrentTermination.productInformation.groupList,productRelation.terminationPreTermination.productInformation.productList,productRelation.terminationPreTermination.productInformation.groupList,productRelation.terminationConcurrentTermination.productInformation.productList,productRelation.terminationConcurrentTermination.productInformation.groupList,productRelation.subRule.productInformation.productList,productRelation.subRule.productInformation.groupList,commonRule.planChangeCondition.dailyPlanChangeLimit.value,commonRule.planChangeCondition.monthlyPlanChangeLimit.value,deductibleInfo.additionalOfferForDisabilities.value,customerInfo.onboardingCustomer.customerTypeRule.eligibility,customerInfo.onboardingCustomer.customerTypeRule.valueList,customerInfo.onboardingCustomer.businessCustomerSubtypeRule.eligibility,customerInfo.onboardingCustomer.businessCustomerSubtypeRule.valueList,otherOnboardInfo.duplicateNameOnboard.domain.valueList,otherOnboardInfo.duplicateNameOnboard.productGroup.productInformation.groupList,topupInfo.chargeAmount.minimumChargeAmount.value,topupInfo.chargeAmount.maximumChargeAmount.value,otherOnboardInfo.specialCustomerOnboard.isSoldier.value,otherOnboardInfo.duplicateNameOnboard.productGroup.eligibility,customerInfo.onboardingCustomer.individualCustomerSubtypeRule.eligibility,customerInfo.onboardingCustomer.individualCustomerSubtypeRule.valueList,benefitOfVoiceCall.performRefill.voiceCallRefillRange.refillAmount,benefitOfVoiceCall.performRefill.voiceCallRefillRange.range,voice.includedVoiceCallVideoOrValueAddedCallSeparateSetting.voiceCallRange.range,voice.includedVoiceCallTospecifiedNumbers.voiceCallRange.range,voice.includedVoiceCallTospecifiedNumbers.voiceCallRange.numberOfService
0,PA00000001,무제한,300분,기본제공,[],15GB,15GB,15GB,[],1Mbps,N,N,38000원,34546원,38000원,후불,N,15GB,Y,2GB,Y,운영,상품 > 기본요금제 > 휴대폰 요금제,다이렉트5G 38,Direct5G 38,다이렉트플랜,"[유심개통, 쓰던폰, USIM개통, 비대면, 자급제, 온라인, 데이터100GB 이하...","[LTE generation, 5G generation]",[NA00007164],월 15GB 데이터를 제공하는 SK텔레콤 공식 온라인 채널인 T다이렉트샵에서만 가입...,T다이렉트샵(SK텔레콤 공식온라인채널)을 통하여 신규/기변한 고객 가입 가능